<a href="https://colab.research.google.com/github/KennedyClydeTIpanag/Final-Project-CS2/blob/main/Dahlia_Hospital_NB.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import requests
import json
url = "https://raw.githubusercontent.com/KennedyClydeTIpanag/Final-Project-CS2/refs/heads/main/data/hospital.json"
response = requests.get(url)
hospital = response.json()
print(hospital)
#make sure the url is the raw file from the github

[{'patient_id': 1, 'name': 'John Cruz', 'age': 45, 'gender': 'Male', 'visits': [{'visit_date': '2025-08-12', 'diagnosis': 'Diabetes Type II', 'treatment': 'Metformin 500mg daily', 'doctor': 'Dr. Santos', 'notes': 'Advised low-carb diet and regular exercise.'}, {'visit_date': '2025-09-10', 'diagnosis': 'Diabetes Type II', 'treatment': 'Increased Metformin dosage to 850mg', 'doctor': 'Dr. Santos', 'notes': 'Blood sugar levels still above target.'}]}, {'patient_id': 2, 'name': 'Ana Gomez', 'age': 32, 'gender': 'Female', 'visits': [{'visit_date': '2025-08-15', 'diagnosis': 'Hypertension Stage 1', 'treatment': 'Losartan 50mg daily', 'doctor': 'Dr. Dela Cruz', 'notes': 'Encouraged lifestyle modifications.'}, {'visit_date': '2025-09-05', 'diagnosis': 'Hypertension Stage 1', 'treatment': 'Continue Losartan 50mg', 'doctor': 'Dr. Dela Cruz', 'notes': 'BP improved; maintain current plan.'}]}, {'patient_id': 3, 'name': 'Luis Ramos', 'age': 60, 'gender': 'Male', 'visits': [{'visit_date': '2025-08-1

In [3]:
!pip install firebase-admin

In [4]:
import firebase_admin
from firebase_admin import credentials, db
# Load the private key
cred = credentials.Certificate("/content/firebase key.json")
# Initialize the app with your database URL
firebase_admin.initialize_app(cred, {
 "databaseURL": "https://dahlia-hospital-db-default-rtdb.asia-southeast1.firebasedatabase.app/"
})
print("Firebase connected successfully!")

Firebase connected successfully!


In [5]:
import requests
import json

data = hospital
print("JSON file loaded")
ref = db.reference("hospital")
for patient_record in data:
  ref.child(str(patient_record["patient_id"])).set(patient_record)
print("Data uploaded successfully!")

JSON file loaded
Data uploaded successfully!


In [ ]:
hospital_ref = db.reference("hospital")
hospital_data = hospital_ref.get()

patient_found = False
input_id = str(input(("Enter a Patient ID (1-4) to find their data: ")))

for patient_value in hospital_data:
  if patient_value is not None and str(patient_value["patient_id"]) == input_id:
    patient_found = True
    age = patient_value["age"]
    gend = patient_value["gender"]
    name = patient_value["name"]

    print("Age: ", age)
    print("Gender:", gend)
    print("Name:", name)

    id2 = int(input("Enter 1 to view their visits: "))
    if id2 == 1:
      visitno = 0
      visits = patient_value["visits"]
      amount = len(visits)
      for visit in visits:
        visitno += 1
        date = visit["visit_date"]
        doctor = visit["doctor"]
        diagnosis = visit["diagnosis"]
        print(f"--- Visit Number: {visitno} ---")
        print("Date: ", date)
        print("Doctor: ", doctor)
        print("Diagnosis: ", diagnosis)
    break # Exit the loop once the patient is found

if not patient_found:
    print(f"Patient with ID {input_id} not found.")


In [8]:
ref = db.reference("hospital")

while True:
    print("\n===== HOSPITAL DATABASE MENU =====")
    print("1. Check Hospital Data")
    print("2. Add Patient")
    print("3. Update Patient Records")
    print("4. Delete Patient Data")
    print("5. Specific Search")
    print("6.  End Program")

    choice = input("Enter choice: ")

    # DISPLAY
    if choice == "1":
        HospitalData = ref.get()
        print("\nHospital Data:")
        if HospitalData:
            for patient in HospitalData:
                if patient is not None:
                    print(f"Patient ID: {patient['patient_id']}, Data status: Intact")
        else:
            print("No hospital data found.")
    # ADD PATIENT
    elif choice == "2":
        print("Adding new patient...")

        # Get all existing patient IDs to determine the next one
        current_patients_data = ref.get()
        if current_patients_data:
            # Filter out None values and extract patient_id
            patient_ids = [p['patient_id'] for p in current_patients_data if p is not None and 'patient_id' in p]
            if patient_ids:
                pat_ID = max(patient_ids) + 1
            else:
                pat_ID = 1 # Start with 1 if no valid patient_ids found
        else:
            pat_ID = 1 # Start with 1 if no patients at all

        print(f"Assigned Patient ID: {pat_ID}")

        name = ""
        while not name.strip():
            name = input("Enter name: ")

        age_str = ""
        while not age_str.strip():
            age_str = input("Enter Age: ")
        age = int(age_str)

        gender = ""
        while not gender.strip():
            gender = input("Enter Gender: ")

        patient_data = {
            "patient_id": int(pat_ID),
            "name": name,
            "age": age,
            "gender": gender,
            "visits": []
        }

        ref.child(str(pat_ID)).set(patient_data)
        print("Patient added successfully!")

    # UPDATE PATIENT RECORD
    elif choice == "3":
        patient_id_input = ""
        while not patient_id_input.strip():
            patient_id_input = input("Enter ID of patient to add visit record: ")
        patient_ref = ref.child(patient_id_input)

        patient_data = patient_ref.get()
        if patient_data:
            visits = patient_data.get('visits', [])

            visit_date = ""
            while not visit_date.strip():
                visit_date = input("Enter visit date: ")

            doctor = ""
            while not doctor.strip():
                doctor = input("Enter doctor's name: ")

            diagnosis = ""
            while not diagnosis.strip():
                diagnosis = input("Enter diagnosis: ")

            treatment = ""
            while not treatment.strip():
                treatment = input("Enter treatment: ")

            notes = ""
            while not notes.strip():
                notes = input("Enter notes: ")

            visit_data = {
                "visit_date": visit_date,
                "doctor": doctor,
                "diagnosis": diagnosis,
                "treatment": treatment,
                "notes": notes
            }

            visits.append(visit_data)
            patient_ref.update({'visits': visits})
            print("Visit record added successfully!")
        else:
            print(f"Patient with ID {patient_id_input} not found.")


    # DELETE
    elif choice == "4":
        pid = ""
        while not pid.strip():
            pid = input("Enter patient ID to delete data: ")

        if ref.child(pid).get():
            ref.child(pid).delete()
            print("Patient deleted successfully!")
        else:
            print("Patient not found.")


    # SPECIFIC SEARCH
    elif choice == "5":
        hospital_data_search = ref.get()

        max_id_for_prompt = 0
        hospital_data_list = []
        if hospital_data_search:
            if isinstance(hospital_data_search, dict):
                hospital_data_list = [v for k, v in hospital_data_search.items() if v is not None]
            else:
                hospital_data_list = [p for p in hospital_data_search if p is not None]

            if hospital_data_list:
                max_id_for_prompt = max(p['patient_id'] for p in hospital_data_list)

        search_id = ""
        while not search_id.strip():
            search_id = input(f"Enter a Patient ID (1-{max_id_for_prompt}) to find their data: ")

        patient_found = False
        counter = 0

        if hospital_data_list: # Use the already processed list
            for patient_value in hospital_data_list:
                if str(patient_value["patient_id"]) == search_id:
                    patient_found = True
                    age = patient_value["age"]
                    gend = patient_value["gender"]
                    name = patient_value["name"]
                    visitlogs = patient_value["visits"]

                    print("\nPatient Name: ", name)
                    print("Age: ", age)
                    print("Gender: ", gend)

                    for x in visitlogs:
                      counter += 1
                      diagnosis = x["diagnosis"]
                      doctor = x["doctor"]
                      date = x["visit_date"]
                      notes = x["notes"]
                      treatment = x["treatment"]
                      print(f"Visit {counter}:")
                      print("    Diagnosis: ", diagnosis)
                      print("    Doctor: ", doctor)
                      print("    Visit Date: ", date)
                      print("    Treatment: ", treatment)
                      print("    Doctor's Notes = ", notes)
                    break # Exit the loop once the patient is found

        if not patient_found:
          print("Patient not found.")


    # EXIT
    elif choice == "6":
        print("Exiting program...")
        break

    else:
        print("Invalid choice. Try again.")

# Loop → keeps menu running
#Conditionals (if-elif-else) → handles choices
#CRUD operations:
#Create → set()
#Read → get()
#Update → update()
#Delete → delete()
#Dictionary (JSON) → data structure
#Firebase interaction


===== HOSPITAL DATABASE MENU =====
1. Check Hospital Data
2. Add Patient
3. Update Patient Records
4. Delete Patient Data
5. Specific Search
6.  End Program
Enter choice: 5
Enter a Patient ID (1-5) to find their data: 5

Patient Name:  Bam Adebayo
Age:  34
Gender:  Male
Visit 1:
    Diagnosis:  Sprained Foot
    Doctor:  Dr. Adams
   Visit Date:  2026-04-20
   Treatment:  Cast, painkillers
   Doctor's Notes =  Proper rest for 2 weeks is needed.

===== HOSPITAL DATABASE MENU =====
1. Check Hospital Data
2. Add Patient
3. Update Patient Records
4. Delete Patient Data
5. Specific Search
6.  End Program
Enter choice: 6
Exiting program...
